# SLAM-LLM ASR (LibriSpeech) - Google Colab Rehberi

Bu notebook, **SLAM-LLM** (Speech, Language, Audio, and Music LLM) mimarisini Google Colab ortamında (T4 GPU veya üstü) çalıştırmak ve test etmek için hazırlanmıştır.

### Referanslar:
- **Paper**: [SLAM-LLM: Structured Language Model for Speech and Audio (arXiv:2402.08846)](https://arxiv.org/abs/2402.08846)
- **GitHub**: [X-LANCE/SLAM-LLM](https://github.com/X-LANCE/SLAM-LLM/tree/main/examples/asr_librispeech)

## Adım 1: GPU Kontrolü ve Bağımlılıkların Kurulumu

In [1]:
# GPU Durumunu Kontrol Et (T4 16GB VRAM minimum önerilir)
!nvidia-smi

# Repozitory klonlama
!git clone https://github.com/X-LANCE/SLAM-LLM.git
%cd SLAM-LLM

# Gerekli kütüphaneleri yükleme
!pip install -e .
!pip install torch torchaudio transformers accelerate peft fairseq soundfile librosa editdistance hydra-core bitsandbytes gdown


/bin/bash: line 1: nvidia-smi: command not found
Cloning into 'SLAM-LLM'...
remote: Enumerating objects: 10526, done.
remote: Counting objects: 100% (2416/2416), done.
remote: Compressing objects: 100% (630/630), done.
remote: Total 10526 (delta 2037), reused 1786 (delta 1786), pack-reused 8110 (from 2)
Receiving objects: 100% (10526/10526), 173.18 MiB | 21.78 MiB/s, done.
Resolving deltas: 100% (6888/6888), done.
/content/SLAM-LLM
Obtaining file:///content/SLAM-LLM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.2/105.2 kB 2.6 MB/s eta 0:00:00
  Using cached pathspec-1.1.1-py3-none-any.whl.metadata (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1

## Adım 2: Pretrained Checkpoint İndirme

SLAM-LLM ASR modelinde yalnızca **Linear Projector** katmanı eğitilmektedir. Makalede sağlanan WavLM Large + Vicuna 7B hazır projector ağırlığını indiriyoruz:

In [2]:
import os
os.makedirs("/content/checkpoints", exist_ok=True)

# WavLM Large + Vicuna 7B Linear Projector (~18.88M)
!gdown --id 1cLNuMR05oXxKj8M_Z3yAZ5JHJ06ybIHp -O /content/checkpoints/wavlm_linear_projector.pt
print("Checkpoint indirildi!")


/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1cLNuMR05oXxKj8M_Z3yAZ5JHJ06ybIHp
From (redirected): https://drive.google.com/uc?id=1cLNuMR05oXxKj8M_Z3yAZ5JHJ06ybIHp&confirm=t&uuid=b80b7dac-4f7b-45f4-8e2f-48f81c5d1d31
To: /content/checkpoints/wavlm_linear_projector.pt
100% 75.5M/75.5M [00:03<00:00, 22.7MB/s]
Checkpoint indirildi!


## Adım 3: Örnek Ses Verisi ve Data Manifest (JSONL) Hazırlama

In [3]:
import json
# Test ses dosyası indirme
!wget -O /content/sample.wav https://www.signalogic.com/melp/Speech_Samples/H264_Test1.wav

manifest_data = [
    {
        "key": "sample_01",
        "source": "/content/sample.wav",
        "target": "this is a test audio file for speech recognition"
    }
]

with open("/content/test.jsonl", "w", encoding="utf-8") as f:
    for entry in manifest_data:
        f.write(json.dumps(entry) + "\n")

print("test.jsonl manifest hazırlandı!")


--2026-10-05 18:27:20--  https://www.signalogic.com/melp/Speech_Samples/H264_Test1.wav
Resolving www.signalogic.com (www.signalogic.com)... 209.150.126.178
Connecting to www.signalogic.com (www.signalogic.com)|209.150.126.178|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-10-05 18:27:21 ERROR 404: Not Found.

test.jsonl manifest hazırlandı!


## Adım 4: Inference (Decode / Ses Tanıma)

In [4]:
!CUDA_VISIBLE_DEVICES=0 python -m slam_llm.pipeline.decode \
    --config-path ../examples/asr_librispeech/config \
    --config-name decode \
    hydra.run.dir=/content/decode_output \
    dataset_config.val_data_path=/content/test.jsonl \
    model_config.llm_name_or_path=lmsys/vicuna-7b-v1.5 \
    model_config.encoder_name_or_path=microsoft/wavlm-large \
    ckpt_path=/content/checkpoints/wavlm_linear_projector.pt


/usr/bin/python3: No module named slam_llm.pipeline.decode


## Adım 5: Kendi Verinizle Eğitme (Fine-Tuning)

In [5]:
!CUDA_VISIBLE_DEVICES=0 python -m slam_llm.pipeline.finetune \
    --config-path ../examples/asr_librispeech/config \
    --config-name finetune \
    dataset_config.train_data_path=/content/test.jsonl \
    dataset_config.val_data_path=/content/test.jsonl \
    train_config.output_dir=/content/train_output \
    train_config.batch_size_training=2 \
    train_config.num_epochs=3 \
    model_config.llm_name_or_path=lmsys/vicuna-7b-v1.5 \
    model_config.encoder_name_or_path=microsoft/wavlm-large


/content/SLAM-LLM/src/slam_llm/utils/train_utils.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/SLAM-LLM/src/slam_llm/pipeline/finetune.py", line 23, in <module>
    from slam_llm.utils import fsdp_auto_wrap_policy
  File "/content/SLAM-LLM/src/slam_llm/utils/__init__.py", line 7, in <module>
    from slam_llm.utils.train_utils import *
  File "/content/SLAM-LLM/src/slam_llm/utils/train_utils.py", line 22, in <module>
    from slam_llm.utils.checkpoint_handler import (
    ...<5 lines>...
    )
  File "/content/SLAM-LLM/src/slam_llm/utils/checkpoint_handler.py", line 9, in <module>
    from deepsp